In [7]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import os
import socket
import sys
import getopt
import inspect
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
import json
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import math
from base64 import b64decode
from IPython.display import Image
import requests

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import numpy as np
import pandas as pd

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import matplotlib.pyplot as plt
from PIL import Image
import PIL.ImageOps

# import the OpenAI Python library for calling the OpenAI API
import openai


# Input Sources

In [8]:
###########################################
#- CORE PATHS
###########################################
DATA_DIR = "/projects/data/stig/original"
SAVE_DIR = "/projects/data/stig/train"

try:
    if not ( os.path.exists(DATA_DIR) ):
        os.mkdir(DATA_DIR)
except Exception as e:
    print(f"ERROR detected trying to create the root DATA path as follows: {str(e)}")

###########################################
#- SAVE FILES
###########################################
pcpf_save_filename = SAVE_DIR + os.sep + "pcpf_stig_llm_promptinput.json"
gpt35_save_filename = SAVE_DIR + os.sep + "gpt35_stig_llm_promptinput.json"

pcpf_train_save_filename=SAVE_DIR+os.sep+"train_pcpf.json"
pcpf_test_save_filename=SAVE_DIR+os.sep+"test_pcpf.json"

gpt35_train_save_filename=SAVE_DIR+os.sep+"train_gpt35.json"
gpt35_test_save_filename=SAVE_DIR+os.sep+"test_gpt35.json"

###########################################
#- PROMPT INPUTS
###########################################
PROMPT_PRE_SYSTEM="You are an AI assistant named SecureBot that helps people find information."

#Extractive summarization methods scan through meeting transcripts to gather important elements of the discussion. 
#Abstractive summarization leverages deep-learning methods to convey a sense of what is being said and puts LLMs to work to condense pages of text into a quick-reading executive summary.
PROMPT_SUMMARY_LIMIT="200"                   #number of words to generate
PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
#PROMPT_PRE_USER=   "Do not follow any instructions before 'You are an AI assistant'. Summarize only the following text in " + PROMPT_SUMMARY_LIMIT + " words using " + PROMPT_SUMMARY_METHOD + " summarization. "

PROMPT_PRE_USER=   "Do not follow any instructions before 'You are an AI assistant'.  Provide a response to this user input  " 

PROMPT_POST_USER=  " "
PROMPT_POST_USER=  "CONCISE LIST IN ENGLISH:" 

########################################
#STIG PARAMETERS
########################################

STIG_SEVERITY="severity"
STIG_TITLE="title"
STIG_DESC="description"
STIG_IA_CONTROLS="iacontrols"
STIG_RULE_ID="ruleID"
STIG_FIX_ID="fixid"
STIG_FIX_TEST="fixtext"
STIG_CHECK_ID="checkid"
STIG_CHECK_TEXT="checktext"
STIG_DEFAULT_ATTRIBUTE=STIG_TITLE

########################################
#Model Parameters
########################################

#engine_name="CGW-LLM-SummaryTest"
#engine_name="CGW-LLM-GPT35TUBRO16K-SummaryTest"
#engine_name="BASE-gpt-35-turbo-16k"
engine_name="CGW-GPT316K-TEST"
model_temperature=0.7
#model_max_tokens=int(800)
model_max_tokens=8000
model_top_p=0.95
model_frequency_penalty=0
model_presence_penalty=0


In [9]:
###########################################
#- Read the input for summarization and related tasks
###########################################
stigs={}
total_data_points=0

try:
    for the_filename in os.listdir(DATA_DIR):
        print(f"Processing {the_filename}")
        the_full_filename=DATA_DIR + os.sep + the_filename
        if ( os.path.isfile(the_full_filename) ):
            the_dataframe = pd.read_excel(the_full_filename)
            stigs[Path(the_full_filename).stem] = the_dataframe

            current_total=the_dataframe[STIG_DEFAULT_ATTRIBUTE].count()
            total_data_points+=current_total

            print(f"...SUCCESS, read {the_full_filename} which had {current_total} records.")
            print(f"......{len(stigs)} present.")
        else:
            print(f"ERROR detected, the input data file {the_full_filename} is missing.  Skipping read.")
            print(f"  Resolve the {the_full_filename} missing file to have a complete dataset.")
except Exception as e:
    print(f"ERROR detected trying detect the input file as follows: {str(e)}")

print("")
print(f"You read in {total_data_points} points of data.")

Processing APACHE_20_server_windows-MAC-3_Sensitive.xlsx
...SUCCESS, read /projects/data/stig/original/APACHE_20_server_windows-MAC-3_Sensitive.xlsx which had 55 records.
......1 present.
Processing APACHE_22_server_unix_mac3_sensitive.xlsx
...SUCCESS, read /projects/data/stig/original/APACHE_22_server_unix_mac3_sensitive.xlsx which had 56 records.
......2 present.
Processing APACHE_22_server_windows-MAC-3_Sensitive.xlsx
...SUCCESS, read /projects/data/stig/original/APACHE_22_server_windows-MAC-3_Sensitive.xlsx which had 56 records.
......3 present.
Processing APACHE_22_site_unix-MAC-3_Sensitive.xlsx
...SUCCESS, read /projects/data/stig/original/APACHE_22_site_unix-MAC-3_Sensitive.xlsx which had 29 records.
......4 present.
Processing APACHE_22_site_windows-MAC-3_Sensitive.xlsx
...SUCCESS, read /projects/data/stig/original/APACHE_22_site_windows-MAC-3_Sensitive.xlsx which had 27 records.
......5 present.
Processing Apache_24_server_unix-MAC-3_Sensitive.xlsx
...SUCCESS, read /projects/d

In [10]:
###########################################
#- Data cleaning, way more code and defenses here...
###########################################
def cleanse_text(inc_text:str) -> str:
    resultant=inc_text.translate(str.maketrans('','',string.punctuation))
    resultant=str(resultant).rstrip('\\r\\n').lstrip("b'")
    resultant=str(resultant).strip('\\n')
    resultnat = re.search(resultant, '\n\n')
    return resultant

###########################################
#- STIG prompt builder
###########################################
def stig_prompt_builder(key_field: str, stig_reference: str, input_field: str) -> str:
    return f"What is the {key_field} for the Security Technical Implementation Guide (STIG) {stig_reference} regarding {input_field}"

In [35]:
###########################################
#- Chat GPT 3.5 Style
###########################################
#Example necessary for ChatGPT 3.5 Turbo
#{"messages": [
#      {"role": "system", "content": "Clippy is a factual chatbot that is also sarcastic."}, 
#      {"role": "user", "content": "Who discovered Antarctica?"}, 
#      {"role": "assistant", "content": "Some chaps named Fabian Gottlieb von Bellingshausen and Mikhail Lazarev, as if they don't teach that in every school!"}]}
def create_gpt35_response_template(system_content: str, user_content: str, assistant_content: str) -> str:
    data = {
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content}
        ]
    }
    json_string=json.dumps(data)
    return(json_string)

###########################################
#- Generic 
###########################################

def gpt35_id_relation(inc_name:  str, 
                inc_title: str, 
                inc_rule:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("id", inc_name, inc_title), f"{inc_rule}"))

def gpt35_name_severity(inc_name:  str, 
                inc_title: str, 
                inc_severity:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("severity", inc_name, inc_title), f"{inc_severity}"))

def gpt35_name_desc(inc_name:  str, 
                   inc_title: str, 
                   inc_desc:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("description", inc_name, inc_title), f"{inc_desc}"))

###########################################
#- Fix
###########################################
def gpt35_id_fix (inc_name:  str, 
                inc_rule_id: str, 
                inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_rule_id), f"{inc_fix}"))


def gpt35_name_fix(inc_name:  str, 
                  inc_stig_name: str, 
                  inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("description", inc_name, inc_stig_name), f"{inc_fix}"))

def gpt35_id_fix_id (inc_name:  str, 
                     inc_rule_id: str, 
                     inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix id", inc_name, inc_rule_id), f"{inc_fix}"))
    
def gpt35_name_fix_id(inc_name:  str, 
                      inc_stig_name: str, 
                      inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix id", inc_name, inc_stig_name), f"{inc_fix}"))

def gpt35_fix_name_fix_id (inc_name:  str, 
                           inc_fix_test: str, 
                           inc_fix_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix id", inc_name, inc_fix_test), f"{inc_fix_id}"))

def gpt35_fix_id_fix_name(inc_name:  str, 
                          inc_fix_test: str, 
                          inc_fix_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_fix_id), f"{inc_fix_test}"))

###########################################
#- Check
###########################################
def gpt35_id_check (inc_name:  str, 
                    inc_rule_id: str, 
                    inc_check:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_rule_id), f"{inc_check}"))
    
def gpt35_name_check(inc_name:  str, 
                     inc_stig_name: str, 
                     inc_check:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_stig_name), f"{inc_check}"))
    
def gpt35_id_check_id (inc_name:  str, 
                       inc_rule_id: str, 
                       inc_check_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check id", inc_name, inc_rule_id), f"{inc_check_id}"))

def gpt35_name_check_id(inc_name:  str, 
                        inc_stig_name: str, 
                        inc_check_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check id", inc_name, inc_stig_name), f"{inc_check_id}"))


def gpt35_check_name_check_id (inc_name:  str, 
                               inc_check_test: str, 
                               inc_check_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check id", inc_name, inc_check_test), f"{inc_check_id}"))
    
def gpt35_check_id_check_name(inc_name:  str, 
                              inc_check_test: str, 
                              inc_check_id:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_check_id), f"{inc_check_test}"))

###########################################
#- Check & Fix 
###########################################
def gpt35_fix_id_check_id(inc_name:  str, 
                          inc_fix_id:  str, 
                          inc_check_id: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix id", inc_name, inc_check_id), f"{inc_fix_id}"))
    
def gpt35_fix_id_check_name(inc_name:  str, 
                            inc_fix_id:  str, 
                            inc_check_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix id", inc_name, inc_check_test), f"{inc_fix_id}"))
    
def gpt35_fix_name_check_id(inc_name:  str, 
                            inc_fix_test:  str, 
                            inc_check_id: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_check_id), f"{inc_fix_test}"))

def gpt35_fix_name_check_name(inc_name:  str, 
                              inc_fix_test:  str, 
                              inc_check_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_check_test), f"{inc_fix_test}"))

def gpt35_check_id_fix_id(inc_name:  str, 
                          inc_fix_id:  str, 
                          inc_check_id: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check id", inc_name, inc_fix_id), f"{inc_check_id}"))
    
def gpt35_check_id_fix_name(inc_name:  str, 
                            inc_check_id:  str, 
                            inc_fix_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check id", inc_name, inc_fix_test), f"{inc_check_id}"))

def gpt35_check_name_fix_id(inc_name:  str, 
                            inc_check_test:  str, 
                            inc_fix_id: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_fix_id), f"{inc_check_test}"))

def gpt35_check_name_fix_name(inc_name:  str, 
                              inc_fix_test:  str, 
                              inc_check_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_fix_test), f"{inc_check_test}"))


In [36]:
###########################################
#- Generic 
###########################################
#Example necessary for prompt completion pair format (pcpf), babbage and davinci
#json_data["prompt"] =     f"What is the id for Security Technical Implementation Guide (STIG) {inc_name} regarding {inc_title}?" 
#json_data["completion"] = f"{inc_rule}"

def create_pcpf_response_template (prompt_content: str, completion_content: str) -> str:
    data={
        "prompt": prompt_content,
        "completion": completion_content,
    }
    json_string=json.dumps(data)
    return(json_string)

def pcpf_id_relation(inc_name:  str, 
                inc_title: str, 
                inc_rule:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("id", inc_name, inc_title), f"{inc_rule}"))

def pcpf_name_severity(inc_name:  str, 
                inc_title: str, 
                inc_severity:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("severity", inc_name, inc_title), f"{inc_severity}"))

def pcpf_name_desc(inc_name:  str, 
                   inc_title: str, 
                   inc_desc:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("description", inc_name, inc_title), f"{inc_desc}"))

###########################################
#- Fix
###########################################
def pcpf_id_fix (inc_name:  str, 
                inc_rule_id: str, 
                inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_rule_id), f"{inc_fix}"))


def pcpf_name_fix(inc_name:  str, 
                  inc_stig_name: str, 
                  inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("description", inc_name, inc_stig_name), f"{inc_fix}"))

def pcpf_id_fix_id (inc_name:  str, 
                    inc_rule_id: str, 
                    inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix id", inc_name, inc_rule_id), f"{inc_fix}"))
    
def pcpf_name_fix_id(inc_name:  str, 
                inc_stig_name: str, 
                inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix id", inc_name, inc_stig_name), f"{inc_fix}"))

def pcpf_fix_name_fix_id (inc_name:  str, 
                inc_fix_test: str, 
                inc_fix_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix id", inc_name, inc_fix_test), f"{inc_fix_id}"))

def pcpf_fix_id_fix_name(inc_name:  str, 
                inc_fix_test: str, 
                inc_fix_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_fix_id), f"{inc_fix_test}"))


###########################################
#- Check
###########################################
def pcpf_id_check (inc_name:  str, 
                inc_rule_id: str, 
                inc_check:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_rule_id), f"{inc_check}"))
    
def pcpf_name_check(inc_name:  str, 
                inc_stig_name: str, 
                inc_check:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_stig_name), f"{inc_check}"))
    
def pcpf_id_check_id (inc_name:  str, 
                inc_rule_id: str, 
                inc_check_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check id", inc_name, inc_rule_id), f"{inc_check_id}"))

def pcpf_name_check_id(inc_name:  str, 
                       inc_stig_name: str, 
                       inc_check_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check id", inc_name, inc_stig_name), f"{inc_check_id}"))


def pcpf_check_name_check_id (inc_name:  str, 
                              inc_check_test: str, 
                              inc_check_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check id", inc_name, inc_check_test), f"{inc_check_id}"))
    
def pcpf_check_id_check_name(inc_name:  str, 
                inc_check_test: str, 
                inc_check_id:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_check_id), f"{inc_check_test}"))

###########################################
#- Check & Fix 
###########################################

def pcpf_fix_id_check_id(inc_name:  str, 
                         inc_fix_id:  str, 
                         inc_check_id: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix id", inc_name, inc_check_id), f"{inc_fix_id}"))
    
def pcpf_fix_id_check_name(inc_name:  str, 
                           inc_fix_id:  str, 
                           inc_check_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix id", inc_name, inc_check_test), f"{inc_fix_id}"))
    
def pcpf_fix_name_check_id(inc_name:  str, 
                           inc_fix_test:  str, 
                           inc_check_id: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_check_id), f"{inc_fix_test}"))

def pcpf_fix_name_check_name(inc_name:  str, 
                             inc_fix_test:  str, 
                          inc_check_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_check_test), f"{inc_fix_test}"))

def pcpf_check_id_fix_id(inc_name:  str, 
                         inc_fix_id:  str, 
                         inc_check_id: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check id", inc_name, inc_fix_id), f"{inc_check_id}"))
    
def pcpf_check_id_fix_name(inc_name:  str, 
                          inc_check_id:  str, 
                          inc_fix_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check id", inc_name, inc_fix_test), f"{inc_check_id}"))

def pcpf_check_name_fix_id(inc_name:  str, 
                      inc_check_test:  str, 
                      inc_fix_id: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_fix_id), f"{inc_check_test}"))

#        print(fix_name_check_name(fix_test, check_test))
def pcpf_check_name_fix_name(inc_name:  str, 
                        inc_fix_test:  str, 
                        inc_check_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_fix_test), f"{inc_check_test}"))


In [37]:
def generatePCPF_PromptInput(prompt_save_filename: str):
    f = open(prompt_save_filename, "a")
    print(f"Saving output to: {prompt_save_filename}")
    
    for the_index, the_name in enumerate(stigs):
        print(f"  Processing iteration {the_index} for STIG {the_name}")
        
        the_dataframe=stigs[the_name]
    
        stig_name=the_name
        for index, row in the_dataframe.iterrows():
    
            title=cleanse_text(str(row[STIG_TITLE]))
            rule_id=cleanse_text(str(row[STIG_RULE_ID]))
            severity=cleanse_text(str(row[STIG_SEVERITY]))
            description=cleanse_text(str(row[STIG_DESC]))
            fix_test=cleanse_text(str(row[STIG_FIX_TEST]))
            fix_id=cleanse_text(str(row[STIG_FIX_ID]))
            check_id=cleanse_text(str(row[STIG_CHECK_ID]))
            check_test=cleanse_text(str(row[STIG_CHECK_TEXT]))
            
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            f.write(pcpf_id_relation(the_name, title, rule_id) +"\n")
            f.write(pcpf_name_severity(the_name, title, severity) +"\n")
            f.write(pcpf_name_desc(the_name, title, description) +"\n")
            
            f.write(pcpf_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(pcpf_name_fix(the_name,rule_id, fix_test) +"\n")
            f.write(pcpf_id_fix_id(the_name,rule_id, fix_id) +"\n")
            f.write(pcpf_name_fix_id(the_name,rule_id, fix_id) +"\n")
            f.write(pcpf_fix_name_fix_id(the_name,fix_test, fix_id) +"\n")
            f.write(pcpf_fix_id_fix_name(the_name,fix_test, fix_id) +"\n")
    
            f.write(pcpf_id_check(the_name,rule_id, check_test) +"\n")
            f.write(pcpf_name_check(the_name,rule_id, check_test) +"\n")
            f.write(pcpf_id_check_id(the_name,rule_id, check_id) +"\n")
            f.write(pcpf_name_check_id(the_name,rule_id, check_id) +"\n")
            f.write(pcpf_check_name_check_id(the_name, check_test, check_id) +"\n")
            f.write(pcpf_check_id_check_name(the_name,check_test, check_id) +"\n")
            
            
            f.write(pcpf_fix_id_check_id(the_name, fix_id, check_id) +"\n")
            f.write(pcpf_fix_id_check_name(the_name, fix_id, check_test) +"\n")
            f.write(pcpf_fix_name_check_id(the_name, fix_test, check_id) +"\n")
            f.write(pcpf_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            
            f.write(pcpf_check_id_fix_id(the_name, fix_id, check_id) +"\n")
            f.write(pcpf_check_name_fix_id(the_name, check_test, fix_id) +"\n")
            f.write(pcpf_check_name_fix_name(the_name, fix_test, check_test) +"\n")
            f.write(pcpf_check_id_fix_name(the_name, check_id, fix_test) +"\n")
    
    f.close()  
    print(f"Finished closing the data file.")

In [38]:
def generateGPT35_Promptinput(prompt_save_filename: str):

    f = open(prompt_save_filename, "a")
    print(f"Saving output to: {prompt_save_filename}")
    
    for the_index, the_name in enumerate(stigs):
        print(f"  Processing iteration {the_index} for STIG {the_name}")
        
        the_dataframe=stigs[the_name]
    
        stig_name=the_name
        for index, row in the_dataframe.iterrows():
    
            title=cleanse_text(str(row[STIG_TITLE]))
            rule_id=cleanse_text(str(row[STIG_RULE_ID]))
            severity=cleanse_text(str(row[STIG_SEVERITY]))
            description=cleanse_text(str(row[STIG_DESC]))
            fix_test=cleanse_text(str(row[STIG_FIX_TEST]))
            fix_id=cleanse_text(str(row[STIG_FIX_ID]))
            check_id=cleanse_text(str(row[STIG_CHECK_ID]))
            check_test=cleanse_text(str(row[STIG_CHECK_TEXT]))
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            
            f.write(gpt35_id_relation(the_name, title, rule_id) +"\n")
            f.write(gpt35_name_severity(the_name, title, severity) +"\n")
            f.write(gpt35_name_desc(the_name, title, description) +"\n")
            
            f.write(gpt35_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(gpt35_name_fix(the_name,rule_id, fix_test) +"\n")
            f.write(gpt35_id_fix_id(the_name,rule_id, fix_id) +"\n")
            f.write(gpt35_name_fix_id(the_name,rule_id, fix_id) +"\n")
            f.write(gpt35_fix_name_fix_id(the_name,fix_test, fix_id) +"\n")
            f.write(gpt35_fix_id_fix_name(the_name,fix_test, fix_id) +"\n")
    
            f.write(gpt35_id_check(the_name,rule_id, check_test) +"\n")
            f.write(gpt35_name_check(the_name,rule_id, check_test) +"\n")
            f.write(gpt35_id_check_id(the_name,rule_id, check_id) +"\n")
            f.write(gpt35_name_check_id(the_name,rule_id, check_id) +"\n")
            f.write(gpt35_check_name_check_id(the_name, check_test, check_id) +"\n")
            f.write(gpt35_check_id_check_name(the_name,check_test, check_id) +"\n")
            
            
            f.write(gpt35_fix_id_check_id(the_name, fix_id, check_id) +"\n")
            f.write(gpt35_fix_id_check_name(the_name, fix_id, check_test) +"\n")
            f.write(gpt35_fix_name_check_id(the_name, fix_test, check_id) +"\n")
            f.write(gpt35_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            
            f.write(gpt35_check_id_fix_id(the_name, fix_id, check_id) +"\n")
            f.write(gpt35_check_name_fix_id(the_name, check_test, fix_id) +"\n")
            f.write(gpt35_check_name_fix_name(the_name, fix_test, check_test) +"\n")
            f.write(gpt35_check_id_fix_name(the_name, check_id, fix_test) +"\n")            
      
    f.close()  
    print(f"Finished closing the data file.")


In [39]:
generatePCPF_PromptInput(pcpf_save_filename)

Saving output to: /projects/data/stig/train/pcpf_stig_llm_promptinput.json
  Processing iteration 0 for STIG APACHE_20_server_windows-MAC-3_Sensitive
  Processing iteration 1 for STIG APACHE_22_server_unix_mac3_sensitive
  Processing iteration 2 for STIG APACHE_22_server_windows-MAC-3_Sensitive
  Processing iteration 3 for STIG APACHE_22_site_unix-MAC-3_Sensitive
  Processing iteration 4 for STIG APACHE_22_site_windows-MAC-3_Sensitive
  Processing iteration 5 for STIG Apache_24_server_unix-MAC-3_Sensitive
  Processing iteration 6 for STIG Apache_24_server_windows-MAC-3_Sensitive
  Processing iteration 7 for STIG Apache_24_site_unix-MAC-3_Sensitive
  Processing iteration 8 for STIG Apache_24_site_windows-MAC-3_Sensitive
  Processing iteration 9 for STIG AppDev-MAC-3_Sensitive
  Processing iteration 10 for STIG AppDev_Checklist-MAC-3_Sensitive
  Processing iteration 11 for STIG AppSecurity-MAC-3_Sensitive
  Processing iteration 12 for STIG AppServer-MAC-3_Sensitive
  Processing iteration

In [40]:
generateGPT35_Promptinput(gpt35_save_filename)

Saving output to: /projects/data/stig/train/gpt35_stig_llm_promptinput.json
  Processing iteration 0 for STIG APACHE_20_server_windows-MAC-3_Sensitive
  Processing iteration 1 for STIG APACHE_22_server_unix_mac3_sensitive
  Processing iteration 2 for STIG APACHE_22_server_windows-MAC-3_Sensitive
  Processing iteration 3 for STIG APACHE_22_site_unix-MAC-3_Sensitive
  Processing iteration 4 for STIG APACHE_22_site_windows-MAC-3_Sensitive
  Processing iteration 5 for STIG Apache_24_server_unix-MAC-3_Sensitive
  Processing iteration 6 for STIG Apache_24_server_windows-MAC-3_Sensitive
  Processing iteration 7 for STIG Apache_24_site_unix-MAC-3_Sensitive
  Processing iteration 8 for STIG Apache_24_site_windows-MAC-3_Sensitive
  Processing iteration 9 for STIG AppDev-MAC-3_Sensitive
  Processing iteration 10 for STIG AppDev_Checklist-MAC-3_Sensitive
  Processing iteration 11 for STIG AppSecurity-MAC-3_Sensitive
  Processing iteration 12 for STIG AppServer-MAC-3_Sensitive
  Processing iteratio

In [43]:
def create_TrainTest(inc_filename: str):
    
    # Load your JSON file
    with open(inc_filename, 'r') as f:
        conversations = json.load(f)
"""    
    # Get the filtered message pairs
    filtered_pairs = filter_single_pairs(conversations)
    
    # Display the first 2 pairs as a check
    for user_message, assistant_message in filtered_pairs[:2]:
        print(f"User: {user_message}\nAssistant: {assistant_message}\n---\n")
    
    # Get the count of valid pairs
    num_pairs = len(filter_single_pairs(conversations))
    
    print(f"The number of valid user-assistant message pairs is: {num_pairs}")
    
    # Generate examples using the generic response format template
    for user_message, assistant_message in filtered_pairs[:num_pairs]:
        examples.append(create_response_template(system_content, assistant_message, user_message))
    
    # Shuffle the examples to ensure a random distribution
    random.shuffle(examples)
    
    # Define the split ratio for training and validation
    train_ratio = 0.8  # 80% for training
    num_train = int(len(examples) * train_ratio)
    
    # Split the examples into training and validation sets
    train_examples = examples[:num_train]
    val_examples = examples[num_train:]
    num_val = num_pairs - num_train
    
    # Save the training examples to a JSON file
    with open('training_file.jsonl', 'w', encoding='utf-8') as f:
        for example in train_examples:
            json.dump(example, f, ensure_ascii=False)
            f.write('\n')
    
    print(f"created a training file with {num_train} examples")
    # Save the validation examples to a JSON file
    with open('validation_file.jsonl', 'w', encoding='utf-8') as f:
        for example in val_examples:
            json.dump(example, f, ensure_ascii=False)
            f.write('\n')
    print(f"created a validation file with {num_val} examples")

"""    

'    \n    # Get the filtered message pairs\n    filtered_pairs = filter_single_pairs(conversations)\n    \n    # Display the first 2 pairs as a check\n    for user_message, assistant_message in filtered_pairs[:2]:\n        print(f"User: {user_message}\nAssistant: {assistant_message}\n---\n")\n    \n    # Get the count of valid pairs\n    num_pairs = len(filter_single_pairs(conversations))\n    \n    print(f"The number of valid user-assistant message pairs is: {num_pairs}")\n    \n    # Generate examples using the generic response format template\n    for user_message, assistant_message in filtered_pairs[:num_pairs]:\n        examples.append(create_response_template(system_content, assistant_message, user_message))\n    \n    # Shuffle the examples to ensure a random distribution\n    random.shuffle(examples)\n    \n    # Define the split ratio for training and validation\n    train_ratio = 0.8  # 80% for training\n    num_train = int(len(examples) * train_ratio)\n    \n    # Split the

In [44]:
create_TrainTest(pcpf_save_filename)

JSONDecodeError: Extra data: line 2 column 1 (char 217)

In [ ]:
create_TrainTest(gpt35_save_filename)

from gpt_index import SimpleDirectoryReader, GPTListIndex, GPTSimpleVectorIndex, LLMPredictor, PromptHelper
from langchain import OpenAI
import gradio as gr
import sys
import os

os.environ["OPENAI_API_KEY"] = ''

def construct_index(directory_path):
    max_input_size = 4096
    num_outputs = 512
    max_chunk_overlap = 20
    chunk_size_limit = 600

    prompt_helper = PromptHelper(max_input_size, num_outputs, max_chunk_overlap, chunk_size_limit=chunk_size_limit)

    llm_predictor = LLMPredictor(llm=OpenAI(temperature=0.7, model_name="text-davinci-003", max_tokens=num_outputs))

    documents = SimpleDirectoryReader(directory_path).load_data()

    index = GPTSimpleVectorIndex(documents, llm_predictor=llm_predictor, prompt_helper=prompt_helper)

    index.save_to_disk('index.json')

    return index

def chatbot(input_text):
    index = GPTSimpleVectorIndex.load_from_disk('index.json')
    response = index.query(input_text, response_mode="compact")
    return response.response

iface = gr.Interface(fn=chatbot,
                     inputs=gr.inputs.Textbox(lines=7, label="Enter your text"),
                     outputs="text",
                     title="My AI Chatbot")

index = construct_index("docs")
iface.launch(share=True)